# 🧪 Prueba del Agente RAG — Globex Corp (sin Streamlit)

Este notebook prueba la lógica del agente (carga de PDFs -> chunks -> embeddings -> FAISS -> LLM)
directamente en celdas, sin necesidad de levantar la interfaz de Streamlit.
Sirve tanto en **Jupyter local** como en **Google Colab**.

**Requisito:** tener tu `GROQ_API_KEY` gratuita de https://console.groq.com/keys


## 1. Instalar dependencias

In [ ]:
# Si estás en Colab o un entorno nuevo, instala las dependencias:
%pip install -q langchain langchain-community langchain-groq faiss-cpu sentence-transformers pypdf reportlab python-dotenv


## 2. Obtener los documentos

Si ya tienes la carpeta `documentos/` con los PDFs (porque corriste los scripts
`generar_pdf_dummy.py` y `generar_documentos_ecommerce.py`), pasa al paso 3.

Si estás en Colab y NO tienes los archivos del proyecto, sube manualmente
`generar_pdf_dummy.py` y `generar_documentos_ecommerce.py` con el panel de
archivos (📁 a la izquierda) y ejecuta la celda de abajo para generarlos.

In [ ]:
import os

if not os.path.exists("documentos") or len(os.listdir("documentos")) == 0:
    # Ejecuta los generadores solo si aún no existen los PDFs
    get_ipython().system('python generar_pdf_dummy.py')
    get_ipython().system('python generar_documentos_ecommerce.py')
else:
    print("Los documentos ya existen:", os.listdir("documentos"))


## 3. Ingresar tu GROQ_API_KEY

In [ ]:
import getpass
import os

# Pide la clave de forma oculta (no queda visible en el notebook)
os.environ["GROQ_API_KEY"] = getpass.getpass("Pega tu GROQ_API_KEY: ")


## 4. Cargar y trocear (chunk) los documentos PDF

In [ ]:
import glob
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

rutas_pdf = glob.glob("documentos/*.pdf")
print(f"PDFs encontrados: {len(rutas_pdf)}")
for r in rutas_pdf:
    print(" -", r)

documentos = []
for ruta in rutas_pdf:
    documentos.extend(PyPDFLoader(ruta).load())

splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=300)
fragmentos = splitter.split_documents(documentos)

print(f"\nTotal de fragmentos (chunks) generados: {len(fragmentos)}")
print("\nEjemplo del primer fragmento:\n")
print(fragmentos[0].page_content[:400])


## 5. Crear embeddings y el índice vectorial FAISS

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(fragmentos, embeddings)

print("✅ Índice FAISS creado en memoria.")

# Prueba rápida de búsqueda semántica (sin LLM todavía)
resultados = vectorstore.similarity_search("dias de vacaciones", k=2)
for doc in resultados:
    print("---")
    print(doc.page_content[:300])


## 6. Armar la cadena conversacional con Groq (LLM)

In [ ]:
from langchain_groq import ChatGroq
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

memoria = ConversationBufferMemory(
    memory_key="chat_history", return_messages=True, output_key="answer"
)

prompt_qa = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Eres el asistente virtual de Globex Corp. Responde SIEMPRE en "
        "español, de forma clara y concisa, basándote únicamente en el "
        "siguiente contexto. Si no está en el contexto, dilo explícitamente.\n\n"
        "Contexto:\n{context}\n\nPregunta: {question}\nRespuesta:"
    ),
)

cadena_rag = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memoria,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": prompt_qa},
)

print("✅ Cadena RAG lista para recibir preguntas.")


## 7. Probar preguntas (edita `pregunta` y vuelve a correr esta celda)

In [ ]:
pregunta = "¿Cuántos días de vacaciones me corresponden con 6 años de antigüedad?"

resultado = cadena_rag.invoke({"question": pregunta})

print("PREGUNTA:", pregunta)
print("\nRESPUESTA:\n", resultado["answer"])

print("\n📎 Fuentes usadas:")
for doc in resultado["source_documents"]:
    print(" -", doc.metadata.get("source"), "| página", doc.metadata.get("page", "?") + 1)


## 8. Chat interactivo por consola (opcional)

Corre esta celda para hacer varias preguntas seguidas sin editar código.
Escribe `salir` para terminar.

In [ ]:
while True:
    pregunta = input("Tu pregunta ('salir' para terminar): ")
    if pregunta.strip().lower() == "salir":
        break
    resultado = cadena_rag.invoke({"question": pregunta})
    print("\n🤖", resultado["answer"], "\n")
